# ASH-KV: Isolated INT8 Kernel Benchmark (Production Architecture)
This notebook isolates the EXACT production Triton INT8 compression kernels from the main ASH-KV codebase. 
It benchmarks using the actual 2D token-aware layout (num_tokens x hidden_dim) and per-token scaling factors.

In [ ]:
!pip install pandas matplotlib


In [ ]:
import torch
import triton
import triton.language as tl

# ---------------------------------------------------------------------------
# 1. TRITON KERNELS (Exact Production Version - Looped)
# ---------------------------------------------------------------------------

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 64}, num_warps=2),
        triton.Config({'BLOCK_SIZE': 64}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 64}, num_warps=8),
        triton.Config({'BLOCK_SIZE': 128}, num_warps=2),
        triton.Config({'BLOCK_SIZE': 128}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 128}, num_warps=8),
        triton.Config({'BLOCK_SIZE': 256}, num_warps=2),
        triton.Config({'BLOCK_SIZE': 256}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 256}, num_warps=8),
    ],
    key=['hidden_dim']
)
@triton.jit
def _int8_encode_kernel(
    bf16_ptr,        # *bfloat16, shape (num_tokens, hidden_dim)
    int8_ptr,        # *int8, shape (num_tokens, hidden_dim)
    scale_ptr,       # *bfloat16, shape (num_tokens,)
    num_tokens,
    hidden_dim,
    BLOCK_SIZE: tl.constexpr,
):
    token_id = tl.program_id(0)

    # Pass 1: Find absolute max across all dimensions
    abs_max = 0.0
    for i in range(0, hidden_dim, BLOCK_SIZE):
        offsets = i + tl.arange(0, BLOCK_SIZE)
        mask = offsets < hidden_dim
        vals = tl.load(bf16_ptr + token_id * hidden_dim + offsets, mask=mask, other=0.0)
        local_max = tl.max(tl.abs(vals), axis=0)
        abs_max = tl.maximum(abs_max, local_max)

    abs_max = tl.maximum(abs_max, 1e-8)
    tl.store(scale_ptr + token_id, abs_max)

    # Pass 2: Quantize and Store
    for i in range(0, hidden_dim, BLOCK_SIZE):
        offsets = i + tl.arange(0, BLOCK_SIZE)
        mask = offsets < hidden_dim
        vals = tl.load(bf16_ptr + token_id * hidden_dim + offsets, mask=mask, other=0.0)
        
        scaled = vals * 127.0 / abs_max
        rounded = scaled + tl.where(scaled > 0.0, 0.5, -0.5)
        int8_vals = tl.cast(rounded, tl.int8)
        tl.store(int8_ptr + token_id * hidden_dim + offsets, int8_vals, mask=mask)

def compress_int8_production(x, hidden_dim):
    x = x.contiguous()
    num_tokens = x.shape[0]
    
    out = torch.empty_like(x, dtype=torch.int8)
    scales = torch.empty((num_tokens,), dtype=x.dtype, device=x.device)
    
    # One block strictly processes one token
    grid = (num_tokens,)
    
    _int8_encode_kernel[grid](x, out, scales, num_tokens, hidden_dim)
    return out, scales


In [ ]:
# ---------------------------------------------------------------------------
# 2. PERFORMANCE BENCHMARKING (GB/s)
# ---------------------------------------------------------------------------

@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['num_tokens'],  # Using num_tokens as x-axis
        x_vals=[2**i for i in range(1, 16, 1)],  # From 2 tokens up to 32K tokens
        x_log=True,
        line_arg='hidden_dim',
        line_vals=[128, 4096],
        line_names=['dim=128', 'dim=4096'],
        styles=[('blue', '-'), ('red', '-')],
        ylabel='GB/s',
        plot_name='prod-int8-compression-performance',
        args={},
    )
)
def benchmark(num_tokens, hidden_dim):
    # Create a tensor mimicking a real KV cache block [num_tokens, hidden_dim]
    x = torch.randn((num_tokens, hidden_dim), device='cuda', dtype=torch.float16)
    quantiles = [0.5, 0.2, 0.8]
    
    ms, min_ms, max_ms = triton.testing.do_bench(lambda: compress_int8_production(x, hidden_dim), quantiles=quantiles)
    
    # Calculate Memory Bandwidth (GB/s)
    # We read FP16 (2 bytes) and write INT8 (1 byte) = 3 bytes per element
    gbps = lambda ms: 3 * x.numel() / (ms * 1e6)
    return gbps(ms), gbps(max_ms), gbps(min_ms)

print("Starting Looped Production Kernel Benchmark...")
benchmark.run(print_data=True, show_plots=True)
